<div style="border-radius:2px; overflow:hidden; border:1px solid #dee2e6;">
  <div style="font-weight:600; padding:10px 14px; border-bottom:1px solid #bee5eb; background-color:#d1ecf1; color:#0c5460;">Learning objectives</div>
  <div style="padding:14px;"><p>Calculate MACE energies for Cu cells at several volumes.</p><p>Fit an equation of state to estimate equilibrium volume and bulk modulus.</p><p>Interpret the energy&#8211;volume curve and convert the bulk modulus to GPa.</p></div>
</div>

# Exercise:  Cu equation of state

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent.parent))

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent.parent))
import numpy as np
import matplotlib.pyplot as plt

from ase.build import bulk
from ase.eos import EquationOfState
from ase.units import kJ

from mace.calculators import MACECalculator

# https://mace-docs.readthedocs.io/en/latest/guide/foundation_models.html
#  MODEL is MACE-MP-0 medium
MODEL = '../../models/2023-12-03-mace-128-L1_epoch-199.model'
calculator = MACECalculator(model_paths=MODEL, 
                            device='cpu',   # 'cuda' for GPU
                            default_dtype='float64')


# Initial crystal structure
atoms = bulk('Cu', 'fcc', a=3.615)
atoms.calc = calculator 
from demo_tools.jhtml import JHTML


<div style="border-radius:2px; overflow:hidden; border:1px solid #dee2e6;">
  <div style="font-weight:600; padding:10px 14px; border-bottom:1px solid #bee5eb; background-color:#d1ecf1; color:#0c5460;">Equation of state workflow</div>
  <div style="padding:14px;">The calculator supplies an energy for each uniformly scaled Cu cell. <code>EquationOfState</code> then fits energy versus volume to estimate the equilibrium volume and bulk modulus.</div>
</div>

In [ ]:

# Save the original cell
cell0 = atoms.cell.copy()

print(f'Lattice vectors of the original cell:\n{cell0}')

<div role="alert" class="alert alert-warning">Keep a copy of the original cell: each scale is defined relative to cell0, not the cell from the previous loop iteration.</div>

In [ ]:

volumes = []
energies = []

# Uniformly scale the lattice vectors
# scale=1 corresponds to the initial lattice parameter
for scale in np.linspace(0.94, 1.06,  9):

    atoms.set_cell(cell0 * scale, scale_atoms=True)

    V = atoms.get_volume()
    E = atoms.get_potential_energy()

    volumes.append(V)
    energies.append(E)

    print(f"scale = {scale:.4f}, V = {V:.4f} Å^3, E = {E:.6f} eV")

# Fit E(V)
eos = EquationOfState(volumes, energies, eos='birchmurnaghan')

V0, E0, B0 = eos.fit()

# ASE returns bulk modulus in eV / Å^3
B0_GPa = B0 / kJ * 1.0e24

print()
print(f"Equilibrium volume = {V0:.6f} Å^3")
print(f"Minimum energy     = {E0:.6f} eV")
print(f"Bulk modulus       = {B0_GPa:.3f} GPa")
print(f" Experimental Bulk modulus = 140 GPa (https://en.wikipedia.org/wiki/Copper)")

# Plot EOS
eos.plot()
plt.show()

<div role="alert" class="alert alert-success">The fit completed. The reported bulk modulus was converted from eV/&#197;&#179; to GPa; inspect the plotted points before interpreting the fit.</div>

# Summary


<div style="border-radius:2px; overflow:hidden; border:1px solid #dee2e6;">
  <div style="font-weight:600; padding:10px 14px; border-bottom:1px solid #c3e6cb; background-color:#d4edda; color:#155724;">What you learned</div>
  <div style="padding:14px;"><p>You sampled a Cu energy&#8211;volume curve with an ASE calculator and fitted its minimum.</p><p>The fitted equilibrium volume and bulk modulus summarize the predicted response to compression.</p></div>
</div>